# Watch a deep_reasoner run from an ACP client

`dr-acp` runs deep_reasoner as an [ACP](https://agentclientprotocol.com) agent: any ACP client
can start it, send it a question, and watch the run arrive as a tree of sessions, one per agent,
each with its cells, its cost and a Stop. This page drives it from Python with
`agent-client-protocol` 0.12.1, on one small question to OpenAI's gpt-6-luna.

## TL;DR

**Definitions.** **`dr-acp`** is the command your ACP client starts; it speaks JSON-RPC over its
stdio. A **conversation** is one ACP session: its first message starts a **run** (one deep_reasoner
REPL) and later messages continue it. A **cell** is one block of Python an agent ran, sent as a tool
call on that agent's session. A **sub-agent** is an agent a cell started; a client that advertises
`subagents` gets each one as a **child session**, announced on its parent's session and placed under
the cell that started it by `_meta.openhands.parentToolCallId`. A **branch** is a sub-agent and every
agent under it. A client without that capability gets the **flat** stream: the same cells on the
root session, labelled with their agents' path (`#1 › #2 › Run …`).

**The takeaway.** You send one question and get the run back as a tree you can rebuild from the
stream alone: every agent's cells, every sub-agent under the cell that started it, and on every
session a cost that already includes everything below it. Stop on one sub-agent ends its branch at
the agents' next turns while the root carries on; until deep_reasoner has a stop of its own, it also
ends the sub-agents running beside it in the same `run_all`. A client without sub-agent support sees
the same cells and can stop only the whole run.

**The evidence.** `/compare-departments Which department is lighter for a first-year student, CS or
STAT?`, on gpt-6-luna: 7 agents, 3 levels deep, 17 model calls.

| session | what its cost covers | cost | x one course agent |
|---|---|---|---|
| root | the whole run | 0.0089 USD | 12.7x |
| n2 | the CS branch: n2's own 3 turns and its 2 course agents | 0.0037 USD | 5.3x |
| n4 | one course agent, 2 turns | 0.0007 USD | 1.0x |

The same question, with Stop sent to the CS department (n2) as soon as it had started a course agent:

| agent | how it ended |
|---|---|
| n2, the CS department | stopped |
| n4, n5, its course agents | stopped by #2, at their next turn |
| n3, the STAT department, beside n2 in the same `run_all` | stopped with #2, once its own course agents had finished |
| root | answered: STAT is lighter |

> **Not yet run.** The code this page drives does not exist yet. Each code cell is followed by a
> cell tagged `expected-output` holding the output written against the design
> (`docs/design/d1-dr-acp.md`), and the TL;DR's numbers are expected values of the same kind. The
> Live Doc Green run executes the page against gpt-6-luna, keeps its real outputs, puts its numbers
> in the TL;DR and deletes every `expected-output` cell, this one included.

## Before you start

You need this repository installed (`uv sync` puts `dr-acp` on your `PATH`), your OpenAI key in
`OPENAI_API_KEY`, and a deep_reasoner config. This page uses `configs/advising/main.yaml`: four courses
in two departments, and one decomposition, `compare departments`, whose program gives each department
to a sub-agent and asks each department to give each of its courses to one more. `prompt.yaml` beside
it holds the system prompt.

In [ ]:
import asyncio, contextlib, os, tempfile
from pathlib import Path

import acp
from deep_reasoning.acp.testing.client import Caps, Printer, ShimConnection, tree

assert "OPENAI_API_KEY" in os.environ, "export OPENAI_API_KEY before you start Jupyter"
config = Path("configs/advising/main.yaml").resolve()
print(config.read_text())

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
# A first-year advisor over a four-course catalog, on OpenAI's gpt-6-luna. Its one
# decomposition, `compare departments`, is a program: dr-acp offers it as the slash
# command /compare-departments. Run it with
#   dr-acp --config docs/configs/advising/main.yaml
_compose:
  - prompt.yaml

model: gpt-6-luna
client:
  base_url: https://api.openai.com/v1
  api_key_env: OPENAI_API_KEY
max_iter: 10
max_depth: 3                 # the root, a department, a course: nothing deeper
entry_namespace: advising

namespaces:
  advising:
    vars:
      catalog:
        CS101:   {title: Programming I,              dept: CS,   credits: 4, hours_per_week: 10, prereqs: []}
        CS102:   {title: Programming II,             dept: CS,   credits: 4, hours_per_week: 12, prereqs: [CS101]}
        STAT101: {title: Introduction to Statistics, dept: STAT, credits: 3, hours_per_week: 6,  prereqs: []}
        STAT102: {title: Probability,                dept: STAT, credits: 3, hours_per_week: 8,  prereqs: [STAT101]}
    decompositions:
      - name: compare departments
        messages:
          - role: user
            content: Which department is lighter for a first-year student?
          - role: assistant
            content: |-
              <think>Each department can be judged on its own, so I give each one to a sub-agent
              and run them together. I read what they found before I answer.</think>
              <repl>
              depts = sorted({c['dept'] for c in catalog.values()})
              found = run_all({d: anext(subagent().send(
                  f'Summarize the workload of the {d} department for a first-year student. '
                  f'Give each {d} course in `catalog` to its own sub-agent with run_all, '
                  f'then combine what they find in two sentences.'))
                  for d in depts})
              print(found)
              </repl>
```

## Start `dr-acp`

`acp.spawn_stdio_transport` starts the agent and hands you its stdio. It gives the subprocess only
`HOME`, `LOGNAME`, `PATH`, `SHELL`, `TERM` and `USER` unless you pass `env=`, and the run needs your
key, so pass your environment. `--home` keeps the run logs in a scratch folder instead of
`~/.deep-reasoning`.

`Caps` and `ShimConnection` let ACP Python 0.12.1 take part in sub-agent sessions, which ACP has only
as unstable types so far: `Caps(subagents={})` puts the capability on the wire, and the connection
hands `subagent_update`, `session_message` and `session_message_chunk` to your client instead of
dropping them. `Printer` is that client: it keeps every update, and one line for each in
`printer.lines`.

In [ ]:
stack = contextlib.AsyncExitStack()                   # keeps every dr-acp below alive until the last cell
home = Path(tempfile.mkdtemp(prefix="dr-home-"))      # run logs
work = Path(tempfile.mkdtemp(prefix="dr-work-"))      # the conversation's working folder
reader, writer, _ = await stack.enter_async_context(acp.spawn_stdio_transport(
    "dr-acp", "--config", str(config), "--home", str(home), env=os.environ))

printer = Printer()
conn = ShimConnection(printer, writer, reader)
init = await conn.initialize(protocol_version=1, client_capabilities=Caps(subagents={}))
init.agent_info

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
Implementation(name='dr-acp', title='deep_reasoner', version='0.1.0', field_meta=None)
```

## Open a conversation

`session/new` answers with the conversation's id and its one option, the namespace. The slash menu,
your namespace's decompositions, follows the response as an update, so wait for it.

In [ ]:
s = await conn.new_session(cwd=str(work), mcp_servers=[])
while not printer.commands:
    await asyncio.sleep(0.05)
namespace = s.config_options[0]
namespace.current_value, [o.value for o in namespace.options], [(c.name, c.description) for c in printer.commands]

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
('advising',
 ['root', 'advising'],
 [('compare-departments', "Open with the 'compare departments' decomposition")])
```

## Ask a question

A message that starts with an advertised command opens the run with that decomposition, and the rest
of the message is the task. Each line below is one ACP update, in the order it arrived: the session it
was sent on (`root`, or `n<node>` for a sub-agent), the update, and what it carried. A cell is
`c<node>.<k>`, the k-th cell of that agent.

In [ ]:
question = "/compare-departments Which department is lighter for a first-year student, CS or STAT?"
seen = len(printer.lines)
r = await conn.prompt(session_id=s.session_id, prompt=[acp.text_block(question)])
print(*printer.lines[seen:], sep="\n")
r.stop_reason, r.field_meta

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
root     available_commands_update  []
root     config_option_update  namespace = advising (fixed)
root     tool_call          c1.1 execute  Run depts = sorted({c['dept'] for c in catalog.values()}) …
root     subagent_update    n2  'Summarize the workload of the CS department for a first-year student. Gi…'  running
root     session_message    → n2  'Summarize the workload of the CS department for a first-year student. …'
root     subagent_update    n3  'Summarize the workload of the STAT department for a first-year student. …'  running
root     session_message    → n3  'Summarize the workload of the STAT department for a first-year studen…'
n2       agent_thought      First I identify the CS courses, then I'll ask a separate sub-agent to summarize…
n2       tool_call          c2.1 execute  Run print([cid for cid, c in catalog.items() if c['dept'] == 'CS'])
n2       tool_call_update   c2.1 completed  ['CS101', 'CS102']
n3       tool_call          c3.1 execute  Run print([(k, v) for k, v in catalog.items() if 'STAT' in str(k) o…
n3       tool_call_update   c3.1 completed  [('STAT101', {'title': 'Introduction to Statistics', 'dept': 'S…
n2       usage_update       0.0004 USD
n3       usage_update       0.0004 USD
root     usage_update       0.0008 USD
n2       agent_thought      There are two CS courses in the catalog. I'll have one sub-agent assess each cour…
n2       tool_call          c2.2 execute  Run summaries = run_all({ …
n2       subagent_update    n4  'Summarize the workload of CS101 for a first-year student.'  running
n2       session_message    → n4  'Summarize the workload of CS101 for a first-year student.'
n2       subagent_update    n5  'Summarize the workload of CS102 for a first-year student.'  running
n2       session_message    → n5  'Summarize the workload of CS102 for a first-year student.'
n4       agent_thought      CS101 is one course, so I'll check its catalog entry directly and report only th…
n4       tool_call          c4.1 execute  Run print(catalog['CS101'])
n4       tool_call_update   c4.1 completed  {'title': 'Programming I', 'dept': 'CS', 'credits': 4, 'hours_p…
n4       agent_thought      CS101 lists 10 hours per week and 4 credits. Its prerequisite list is empty, so …
n4       tool_call          c4.2 execute  Run FinalAnswer('CS101: 4 credits, about 10 hours a week, no prereq…
n4       tool_call_update   c4.2 completed  FinalAnswer: 'CS101: 4 credits, about 10 hours a week, no prereq…
n4       session_message    → n2  'CS101: 4 credits, about 10 hours a week, no prerequisites.'
n4       usage_update       0.0007 USD
n2       subagent_update    n4  idle end_turn
  … n5 the same; n3, with its course agents n6 and n7, likewise, all interleaved …
n2       tool_call_update   c2.2 completed  {'CS101': 'CS101: 4 credits, about 10 hours a week, no prerequi…
n2       agent_thought      CS101 is the more accessible starting point, while CS102 has a slightly higher w…
n2       tool_call          c2.3 execute  Run FinalAnswer('CS runs 10 to 12 hours a week: CS101 is open to a…
n2       tool_call_update   c2.3 completed  FinalAnswer: 'CS runs 10 to 12 hours a week: CS101 is open to a…
n2       session_message    → root  'CS runs 10 to 12 hours a week: CS101 is open to a first-year student, C…'
n2       usage_update       0.0037 USD
root     subagent_update    n2  idle end_turn
  … n3 the same …
root     tool_call_update   c1.1 completed  {'CS': 'CS runs 10 to 12 hours a week: CS101 is open to a first…
root     agent_thought      STAT has lower estimated weekly workloads than CS in the courses listed: 6–8 hou…
root     tool_call          c1.2 execute  Run FinalAnswer("STAT is lighter: its courses take 6 to 8 hours a we…
root     tool_call_update   c1.2 completed  FinalAnswer: "STAT is lighter: its courses take 6 to 8 hours a w…
root     agent_message      STAT is lighter: its courses take 6 to 8 hours a week, CS's 10 to 12.
root     usage_update       0.0089 USD
('end_turn', {'deep_reasoner': {'run': '20261002-162835-3fa9c1', 'outcome': 'answered'}})
```

The first two lines took the menu away and fixed the namespace: a decomposition can only open a run,
and a run keeps its namespace. Each sub-agent was announced with a `subagent_update` on its parent's
session, handed its task in a `session_message`, ran its cells on its own session, sent its answer back
in a `session_message` and went `idle`. The parent's cell completed only when all of them had.

## The tree

`tree` rebuilds the run from the updates alone: each sub-agent under the cell that started it, with its
own cells, how it ended and what it cost.

In [ ]:
print(tree(printer.updates, mode="native"))

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
root  /compare-departments Which department is lighter for a first-year stu…  end_turn · 0.0089 USD
├─ c1.1  Run depts = sorted({c['dept'] for c in catalog.values()}) …  completed → {'CS': 'CS runs 10…
│  ├─ n2  Summarize the workload of the CS department for a first-year student. …  done · 0.0037 USD
│  │  ├─ c2.1  Run print([cid for cid, c in catalog.items() if c['dept'] == 'CS'])  completed → ['CS1…
│  │  ├─ c2.2  Run summaries = run_all({ …  completed → {'CS101': 'CS101: 4 credits, about 10 hours a…
│  │  │  ├─ n4  Summarize the workload of CS101 for a first-year student.  done · 0.0007 USD
│  │  │  │  ├─ c4.1  Run print(catalog['CS101'])  completed → {'title': 'Programming I', 'dept': 'CS',…
│  │  │  │  └─ c4.2  Run FinalAnswer('CS101: 4 credits, about 10 hours a week, no prerequi…  completed
│  │  │  └─ n5  Summarize the workload of CS102 for a first-year student.  done · 0.0008 USD
│  │  │     ├─ c5.1  Run print(catalog['CS102'])  completed → {'title': 'Programming II', 'dept': 'CS'…
│  │  │     └─ c5.2  Run FinalAnswer('CS102: 4 credits, about 12 hours a week, needs CS101 …  completed
│  │  └─ c2.3  Run FinalAnswer('CS runs 10 to 12 hours a week: CS101 is open to a firs…  completed
│  └─ n3  Summarize the workload of the STAT department for a first-year student. …  done · 0.0036 USD
│     ├─ c3.1  Run print([(k, v) for k, v in catalog.items() if 'STAT' in str(k) or …  completed → [('…
│     ├─ c3.2  Run results = run_all({ …  completed → {'STAT101': 'STAT101: 3 credits, about 6 hours…
│     │  ├─ n6  Summarize the workload of STAT101 for a first-year student.  done · 0.0007 USD
│     │  │  ├─ c6.1  Run print(catalog['STAT101'])  completed → {'title': 'Introduction to Statistics'…
│     │  │  └─ c6.2  Run FinalAnswer('STAT101: 3 credits, about 6 hours a week, no prerequ…  completed
│     │  └─ n7  Summarize the workload of STAT102 for a first-year student.  done · 0.0008 USD
│     │     ├─ c7.1  Run print(catalog['STAT102'])  completed → {'title': 'Probability', 'dept': 'STAT…
│     │     └─ c7.2  Run FinalAnswer('STAT102: 3 credits, about 8 hours a week, needs STAT1…  completed
│     └─ c3.3  Run FinalAnswer('STAT runs 6 to 8 hours a week: STAT101 is open to a fi…  completed
└─ c1.2  Run FinalAnswer("STAT is lighter: its courses take 6 to 8 hours a week, CS's…  completed
```

A sub-agent's place comes from its own announcement: `_meta.openhands.parentToolCallId` names a cell its
parent's session has already received, and that is all a client needs to nest it. `_meta.deep_reasoner`
is deep_reasoner's own account of the node.

In [ ]:
n2 = printer.subagents["n2"]
spawned_by = n2.field_meta["openhands"]["parentToolCallId"]
print([(sid, u["title"]) for sid, u in printer.updates
       if u["sessionUpdate"] == "tool_call" and u["toolCallId"] == spawned_by])
n2.field_meta

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
[('s-7c1f9e0a2b4d6e8f', "Run depts = sorted({c['dept'] for c in catalog.values()}) …")]
{'openhands': {'parentToolCallId': '20261002-162835-3fa9c1-n1-c1'},
 'deep_reasoner': {'run': '20261002-162835-3fa9c1',
  'node': 2,
  'parent': 1,
  'depth': 2,
  'namespace': 'advising',
  'backbone': 'chat',
  'drive': 1,
  'status': 'done'}}
```

## What it cost

Every session sends `usage_update`, and its `cost` already includes every agent under it: the root's is
the whole run, and you never add a child's cost to its parent's. gpt-6-luna's API reports tokens, not
prices, so `dr-acp` prices each call from its own table (`cost_source: table`), and while any call has
no price it leaves `cost` out rather than report a total it does not have. `used` is the root's context
at its last call, `size` the model's window.

In [ ]:
[u for sid, u in printer.updates if sid == s.session_id and u["sessionUpdate"] == "usage_update"][-1]

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
{'used': 1514,
 'size': 400000,
 'cost': {'amount': 0.0089, 'currency': 'USD'},
 '_meta': {'deep_reasoner': {'cost_source': 'table',
   'tokens_in': 21352,
   'tokens_out': 2047,
   'unknown_calls': 0}},
 'sessionUpdate': 'usage_update'}
```

## Stop one branch

Every sub-agent's announcement offers `cancel`. `session/cancel` with a sub-agent's session id stops it
and every agent under it at their next turn, before their next model call; a cell that is already
running finishes first. Ask the same question in a second conversation, on a connection of its own so
its tree stays apart from the first one's.

In [ ]:
reader2, writer2, _ = await stack.enter_async_context(acp.spawn_stdio_transport(
    "dr-acp", "--config", str(config), "--home", str(home), env=os.environ))
printer2 = Printer()
conn2 = ShimConnection(printer2, writer2, reader2)
await conn2.initialize(protocol_version=1, client_capabilities=Caps(subagents={}))
s2 = await conn2.new_session(cwd=str(work), mcp_servers=[])
s2.session_id

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
's-0d93b27e5a1c4f68'
```

Send the question without waiting for the answer, and as soon as a department has started a course agent
of its own, stop that department.

In [ ]:
turn = asyncio.create_task(conn2.prompt(session_id=s2.session_id, prompt=[acp.text_block(question)]))
async with asyncio.timeout(120):
    while not any(a.field_meta["deep_reasoner"]["depth"] == 3 for a in printer2.subagents.values()):
        await asyncio.sleep(0.1)
course = next(a for a in printer2.subagents.values() if a.field_meta["deep_reasoner"]["depth"] == 3)
department = printer2.subagents[f"n{course.field_meta['deep_reasoner']['parent']}"]
await conn2.cancel(session_id=department.session_id)
r2 = await turn
print(*printer2.lines, sep="\n")
r2.stop_reason, r2.field_meta

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
root     available_commands_update  ['compare-departments']
root     available_commands_update  []
root     config_option_update  namespace = advising (fixed)
root     tool_call          c1.1 execute  Run depts = sorted({c['dept'] for c in catalog.values()}) …
root     subagent_update    n2  'Summarize the workload of the CS department for a first-year student. Gi…'  running
root     session_message    → n2  'Summarize the workload of the CS department for a first-year student. …'
root     subagent_update    n3  'Summarize the workload of the STAT department for a first-year student. …'  running
root     session_message    → n3  'Summarize the workload of the STAT department for a first-year studen…'
  … both departments look at the catalog …
n2       tool_call          c2.2 execute  Run found = run_all({cid: anext(subagent().send( …
n2       subagent_update    n4  'Summarize the workload of CS101 for a first-year student.'  running
n2       session_message    → n4  'Summarize the workload of CS101 for a first-year student.'
n2       subagent_update    n5  'Summarize the workload of CS102 for a first-year student.'  running
n2       session_message    → n5  'Summarize the workload of CS102 for a first-year student.'
n2       agent_thought      Stop requested: this agent and its sub-agents stop at their next turn, before th…
n4       tool_call          c4.1 execute  Run print(catalog['CS101'])
n4       tool_call_update   c4.1 completed  {'title': 'Programming I', 'dept': 'CS', 'credits': 4, 'hours_p…
n5       tool_call          c5.1 execute  Run print(catalog['CS102'])
n5       tool_call_update   c5.1 completed  {'title': 'Programming II', 'dept': 'CS', 'credits': 4, 'hours_…
n4       usage_update       0.0003 USD
n2       subagent_update    n4  idle cancelled
n5       usage_update       0.0003 USD
n2       subagent_update    n5  idle cancelled
n2       tool_call_update   c2.2 completed  deep_reasoning.acp.worker.stop.StoppedByUser: stopped #2 and its…
n2       usage_update       0.0015 USD
root     subagent_update    n2  idle cancelled
  … n3's course agents n6 and n7 finish; n3 was waiting on them in its cell …
n3       tool_call_update   c3.2 completed  {'STAT101': 'STAT101: 3 credits, about 6 hours a week, no prer…
n3       usage_update       0.0029 USD
root     subagent_update    n3  idle cancelled
root     tool_call_update   c1.1 completed  deep_reasoning.acp.worker.stop.StoppedByUser: stopped #2 and its…
root     agent_thought      The departments' summaries were stopped, so I'll compare the weekly hours in th…
root     tool_call          c1.2 execute  Run print({cid: (c['dept'], c['hours_per_week']) for cid, c in ca…
root     tool_call_update   c1.2 completed  {'CS101': ('CS', 10), 'CS102': ('CS', 12), 'STAT101': ('STAT', …
root     agent_thought      STAT's courses take 6 and 8 hours a week, CS's 10 and 12.
root     tool_call          c1.3 execute  Run FinalAnswer('STAT is lighter: 6 to 8 hours a week against CS…
root     tool_call_update   c1.3 completed  FinalAnswer: 'STAT is lighter: 6 to 8 hours a week against CS…
root     agent_message      STAT is lighter: 6 to 8 hours a week against CS's 10 to 12.
root     usage_update       0.0071 USD
('end_turn', {'deep_reasoner': {'run': '20261002-163410-8b2e07', 'outcome': 'answered'}})
```

The stopped department said at once when the stop would land, and its course agents ended at their next
turn. Until deep_reasoner has a stop of its own, `dr-acp` stops an agent from outside it, and that also
ends the agents running beside it in the same `run_all`: here the other department, once its own course
agents had finished. The root's cell output names both, and the root carried on and answered.

In [ ]:
print(tree(printer2.updates, mode="native"))

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
root  /compare-departments Which department is lighter for a first-year stu…  end_turn · 0.0071 USD
├─ c1.1  Run depts = sorted({c['dept'] for c in catalog.values()}) …  completed
│  │  → deep_reasoning.acp.worker.stop.StoppedByUser: stopped #2 and its branch (#4, #5). Its running
│  │    sibling #3 in this run_all was stopped with it (A3).
│  ├─ n2  Summarize the workload of the CS department for a first-year student. …  stopped · 0.0015 USD
│  │  ├─ c2.1  Run print([cid for cid, c in catalog.items() if c['dept'] == 'CS'])  completed → ['CS1…
│  │  └─ c2.2  Run found = run_all({cid: anext(subagent().send( …  completed → deep_reasoning.acp.work…
│  │     ├─ n4  Summarize the workload of CS101 for a first-year student.  stopped by #2 · 0.0003 USD
│  │     │  └─ c4.1  Run print(catalog['CS101'])  completed → {'title': 'Programming I', 'dept': 'CS',…
│  │     └─ n5  Summarize the workload of CS102 for a first-year student.  stopped by #2 · 0.0003 USD
│  │        └─ c5.1  Run print(catalog['CS102'])  completed → {'title': 'Programming II', 'dept': 'CS'…
│  └─ n3  Summarize the workload of the STAT department for a first-year student. …  stopped with #2 · 0.0029 USD
│     ├─ c3.1  Run print([cid for cid, c in catalog.items() if c['dept'] == 'STAT'])  completed → ['…
│     └─ c3.2  Run results = run_all({ …  completed → {'STAT101': 'STAT101: 3 credits, about 6 hours…
│        ├─ n6  Summarize the workload of STAT101 for a first-year student.  done · 0.0007 USD
│        │  ├─ c6.1  Run print(catalog['STAT101'])  completed → {'title': 'Introduction to Statistics'…
│        │  └─ c6.2  Run FinalAnswer('STAT101: 3 credits, about 6 hours a week, no prerequ…  completed
│        └─ n7  Summarize the workload of STAT102 for a first-year student.  done · 0.0008 USD
│           ├─ c7.1  Run print(catalog['STAT102'])  completed → {'title': 'Probability', 'dept': 'STAT…
│           └─ c7.2  Run FinalAnswer('STAT102: 3 credits, about 8 hours a week, needs STAT1…  completed
├─ c1.2  Run print({cid: (c['dept'], c['hours_per_week']) for cid, c in ca…  completed → {'CS101': ('CS'…
└─ c1.3  Run FinalAnswer('STAT is lighter: 6 to 8 hours a week against CS's 10 …  completed
```

## A client without sub-agent support

A stock client (stock OpenHands, Zed) advertises no `subagents`, so `dr-acp` sends the same work on the
root session: each sub-agent is one more tool call (kind `other`) that completes with its answer, and
each cell is labelled with the path of agents that ran it, `#1 › #2 ›`. Such a client never sees a
sub-agent's session, so the only Stop it has is the whole run's. Connect with ACP Python's own
`ClientSideConnection`, no shim and no `Caps`:

In [ ]:
reader3, writer3, _ = await stack.enter_async_context(acp.spawn_stdio_transport(
    "dr-acp", "--config", str(config), "--home", str(home), env=os.environ))
printer3 = Printer()
conn3 = acp.client.ClientSideConnection(printer3, writer3, reader3)
await conn3.initialize(protocol_version=1)
s3 = await conn3.new_session(cwd=str(work), mcp_servers=[])
s3.session_id

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
's-5be1a07c93d24f16'
```

Ask the same question, and stop the whole run once a course agent has appeared.

In [ ]:
turn = asyncio.create_task(conn3.prompt(session_id=s3.session_id, prompt=[acp.text_block(question)]))
async with asyncio.timeout(120):
    while not any(u["sessionUpdate"] == "tool_call" and u.get("kind") == "other"
                  and u["_meta"]["deep_reasoner"]["depth"] == 3 for _, u in printer3.updates):
        await asyncio.sleep(0.1)
await conn3.cancel(session_id=s3.session_id)
r3 = await turn
print(*printer3.lines, sep="\n")
r3.stop_reason

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
root     available_commands_update  ['compare-departments']
root     available_commands_update  []
root     config_option_update  namespace = advising (fixed)
root     tool_call          c1.1 execute  Run depts = sorted({c['dept'] for c in catalog.values()}) …
root     tool_call          a2.1 other    #1 › #2 · Summarize the workload of the CS department for a first…
root     tool_call          a3.1 other    #1 › #3 · Summarize the workload of the STAT department for a fir…
root     tool_call          c2.1 execute  #1 › #2 › Run print([cid for cid, c in catalog.items() if c['dept…
root     tool_call_update   c2.1 completed  ['CS101', 'CS102']
root     tool_call          c3.1 execute  #1 › #3 › Run print([(k, v) for k, v in catalog.items() if 'STAT…
root     tool_call_update   c3.1 completed  [('STAT101', {'title': 'Introduction to Statistics', 'dept': 'S…
root     usage_update       0.0008 USD
root     tool_call          c2.2 execute  #1 › #2 › Run summaries = run_all({ …
root     tool_call          a4.1 other    #1 › #2 › #4 · Summarize the workload of CS101 for a first-year s…
root     tool_call_update   a4.1 failed  Not finished: the run was stopped
root     tool_call_update   c2.2 failed  Not finished: the run was stopped
root     tool_call_update   a2.1 failed  Not finished: the run was stopped
root     tool_call_update   a3.1 failed  Not finished: the run was stopped
root     tool_call_update   c1.1 failed  Not finished: the run was stopped
root     agent_message      Stopped. The run was ended and its REPL state is gone: deep_reasoner cannot cancel a
                            running cell yet (ask A3). Your next message starts a fresh run.
root     usage_update       0.0011 USD
'cancelled'
```

Stopping the run ends deep_reasoner's process, so the REPL's variables and sub-agents are gone. The
conversation itself stays usable, and its next message says it starts afresh:

In [ ]:
seen = len(printer3.lines)
r3 = await conn3.prompt(session_id=s3.session_id,
                        prompt=[acp.text_block("Which courses can a first-year student take with no prerequisites?")])
print(*printer3.lines[seen:], sep="\n")
r3.stop_reason

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
root     agent_message      Starting a fresh run: the previous one was stopped, so its variables and sub-agents
                            are gone.
root     agent_thought      I'll look for the courses whose prerequisite list is empty.
root     tool_call          c1.1 execute  Run print([cid for cid, c in catalog.items() if not c['prereqs']])
root     tool_call_update   c1.1 completed  ['CS101', 'STAT101']
root     agent_thought      CS101 and STAT101 have no prerequisites.
root     tool_call          c1.2 execute  Run FinalAnswer(['CS101', 'STAT101'])
root     tool_call_update   c1.2 completed  FinalAnswer: ['CS101', 'STAT101']
root     agent_message      ['CS101', 'STAT101']
root     usage_update       0.0014 USD
'end_turn'
```

## When it goes wrong

A decomposition only opens a conversation. Sent later, it is answered, and nothing runs:

In [ ]:
seen = len(printer.lines)
r = await conn.prompt(session_id=s.session_id,
                      prompt=[acp.text_block("/compare-departments Which is lighter for a part-time student?")])
print(*printer.lines[seen:], sep="\n")
r.stop_reason, r.field_meta

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
root     agent_message      /compare-departments starts a conversation with a decomposition, so it only works as
                            the first message. Start a new conversation, or ask in plain words.
root     usage_update       0.0089 USD
('end_turn', {'deep_reasoner': {'run': None, 'outcome': 'rejected'}})
```

Started without `env=`, `dr-acp` gets the six variables `spawn_stdio_transport` passes by default and
not your key. The run fails before its first model call, and the message says why; pass `env=os.environ`
as above.

In [ ]:
reader4, writer4, _ = await stack.enter_async_context(acp.spawn_stdio_transport(
    "dr-acp", "--config", str(config), "--home", str(home)))
printer4 = Printer()
conn4 = ShimConnection(printer4, writer4, reader4)
await conn4.initialize(protocol_version=1, client_capabilities=Caps(subagents={}))
s4 = await conn4.new_session(cwd=str(work), mcp_servers=[])
r4 = await conn4.prompt(session_id=s4.session_id, prompt=[acp.text_block(question)])
print(*printer4.lines, sep="\n")
r4.stop_reason, r4.field_meta

**Expected output** (written against the design; the Live Doc Green run replaces it)

```text
root     available_commands_update  ['compare-departments']
root     available_commands_update  []
root     config_option_update  namespace = advising (fixed)
root     agent_message      Could not start the run: ValueError: Missing API key. Set `OPENAI_API_KEY`
                            (preferred) or `OPENAI_API_KEY`.
root     usage_update       0.0000 USD
('end_turn', {'deep_reasoner': {'run': '20261002-163655-c41d2a', 'outcome': 'build_failed'}})
```

## Clean up

Closing the stack closes each `dr-acp`'s stdin; each one closes its conversations, ends their runs and
exits. The run logs stay in `home`.

In [ ]:
await stack.aclose()